In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:

from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

print("========== DATASET STRUCTURE ==========")

for folder in INPUT_DIR.iterdir():
    print("\nDataset:", folder)

    for item in folder.iterdir():
        print("  ", item.name)


========== DATASET STRUCTURE ==========

Dataset: /kaggle/input/datasets
   ammritkumaar


In [5]:

from pathlib import Path

# Dataset path
DATASET = Path(
    "/kaggle/input/datasets/ammritkumaar/rdd2022-preprocessed-yolo"
)

print("Dataset exists:", DATASET.exists())

assert DATASET.exists(), "Dataset path not found!"

# Check required files and folders
print("\n========== DATASET STRUCTURE ==========")

for item in DATASET.iterdir():
    print(item.name)

# Verify image and label counts
print("\n========== DATASET VERIFICATION ==========")

for split in ["train", "val", "test"]:

    images = list((DATASET / "images" / split).glob("*.jpg"))
    labels = list((DATASET / "labels" / split).glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))

# Check YAML
print("\ndata.yaml exists:", (DATASET / "data.yaml").exists())


Dataset exists: True

========== DATASET STRUCTURE ==========
data.yaml
images
labels

========== DATASET VERIFICATION ==========

TRAIN
Images: 16636
Labels: 16636

VAL
Images: 4753
Labels: 4753

TEST
Images: 2377
Labels: 2377

data.yaml exists: True


In [6]:

import yaml
from pathlib import Path

# Kaggle dataset path
DATASET = Path(
    "/kaggle/input/datasets/ammritkumaar/rdd2022-preprocessed-yolo"
)

# New YAML configuration
data_config = {
    "path": str(DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 4,
    "names": {
        0: "Longitudinal Crack",
        1: "Transverse Crack",
        2: "Alligator Crack",
        3: "Pothole"
    }
}

# Save YAML in writable directory
YAML_PATH = Path("/kaggle/working/data.yaml")

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

# Verify YAML
print("========== KAGGLE DATA YAML ==========")

print(YAML_PATH.read_text())

print("YAML saved:", YAML_PATH)
print("YAML exists:", YAML_PATH.exists())


========== KAGGLE DATA YAML ==========
path: /kaggle/input/datasets/ammritkumaar/rdd2022-preprocessed-yolo
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: Longitudinal Crack
  1: Transverse Crack
  2: Alligator Crack
  3: Pothole

YAML saved: /kaggle/working/data.yaml
YAML exists: True


In [7]:

# ==============================
# CELL 4: Install Ultralytics & GPU Check
# ==============================

!pip install -q ultralytics

import torch
import ultralytics
from pathlib import Path

print("========== ENVIRONMENT CHECK ==========")

print("PyTorch Version:", torch.__version__)
print("Ultralytics Version:", ultralytics.__version__)

print("\nCUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):

    print(f"\nGPU {i}:", torch.cuda.get_device_name(i))

    memory = torch.cuda.get_device_properties(i).total_memory

    print("GPU Memory:", round(memory / 1024**3, 2), "GB")

# Check YAML
YAML_PATH = Path("/kaggle/working/data.yaml")

print("\nData YAML exists:", YAML_PATH.exists())

assert torch.cuda.is_available(), "GPU not enabled!"
assert YAML_PATH.exists(), "Run Cell 3 again!"

print("\nEnvironment ready for YOLO training!")


========== ENVIRONMENT CHECK ==========
PyTorch Version: 2.11.0+cu130
Ultralytics Version: 8.4.211

CUDA Available: True
GPU Count: 2

GPU 0: Tesla T4
GPU Memory: 14.56 GB

GPU 1: Tesla T4
GPU Memory: 14.56 GB

Data YAML exists: True

Environment ready for YOLO training!


# 02 — Train Baseline YOLOv8n

## Aim
To train a baseline YOLOv8n model using the SGD optimizer
on the preprocessed RDD2022 dataset for road damage detection.

## Configuration
- Model: YOLOv8n
- Optimizer: SGD
- Epochs: 10
- Image Size: 640 × 640
- Batch Size: 32
- Training Images: 16,636
- Validation Images: 4,753
- Hardware: NVIDIA Tesla T4 x2

## Expected Outputs
- Trained model weights (best.pt and last.pt)
- Training and validation loss curves
- Precision, Recall, mAP50 and mAP50-95
- Training results CSV

In [9]:

# ==========================================
# NOTEBOOK 02 - CELL 5
# Train Baseline YOLOv8n using SGD
# ==========================================

from ultralytics import YOLO
from pathlib import Path
import torch
import time

# 1. Configuration
YAML_PATH = "/kaggle/working/data.yaml"
PROJECT_DIR = "/kaggle/working/YOLO_Experiments"

assert Path(YAML_PATH).exists(), "Run Cell 3 first!"
assert torch.cuda.device_count() == 2, "Enable T4 x2 GPU!"

# 2. Load pretrained model
model = YOLO("yolov8n.pt")

print("========== BASELINE TRAINING ==========")
print("Model: YOLOv8n")
print("Optimizer: SGD")
print("Epochs: 50")
print("Training images: 16,636")
print("Validation images: 4,753")
print("GPU: Tesla T4 x2")
print("Checkpoint: Every 5 epochs")

start_time = time.time()

# 3. Train model
results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=32,
    device=[0, 1],
    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    cos_lr=True,
    mosaic=0.0,
    seed=42,
    workers=2,
    cache=False,
    amp=True,
    patience=15,
    save=True,
    save_period=5,
    plots=True,
    project=PROJECT_DIR,
    name="baseline_yolov8n_sgd",
    exist_ok=False,
    verbose=True
)

# 4. Training summary
elapsed = (time.time() - start_time) / 60

print("\n========== TRAINING COMPLETED ==========")
print("Training time:", round(elapsed, 2), "minutes")
print("Results directory:", model.trainer.save_dir)
print("Best model:", model.trainer.best)
print("Last model:", model.trainer.last)


========== BASELINE TRAINING ==========
Model: YOLOv8n
Optimizer: SGD
Epochs: 50
Training images: 16,636
Validation images: 4,753
GPU: Tesla T4 x2
Checkpoint: Every 5 epochs
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.61G      2.076      2.442      1.792         28        640: 100% ━━━━━━━━━━━━ 520/520 2.9it/s 3:010.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 2.1it/s 36.5s0.3ss
                   all       4752      11016      0.343      0.284      0.234     0.0939

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.61G      2.149      2.522      1.854         20        640: 100% ━━━━━━━━━━━━ 520/520 2.9it/s 3:000.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 2.1it/s 36.3s0.3ss
                   all       4752      11016      0.191      0.207     0.0963      0.035

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.61G      2.198      2.597      1.958         40       

W1113 07:33:25.064000 211 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W1113 07:33:25.065000 211 torch/distributed/elastic/multiprocessing/api.py:1012] Sending process 214 closing signal SIGINT
W1113 07:33:25.078000 211 torch/distributed/elastic/multiprocessing/api.py:1012] Sending process 215 closing signal SIGINT
[rank1]: Exception in thread Thread-13 (_pin_memory_loop):
[rank1]: Traceback (most recent call last):
[rank1]:   File "/root/.config/Ultralytics/DDP/_temp_c8vh7ntd134183668113936.py", line 16, in <module>
[rank1]:     trainer.train()
[rank1]:     ~~~~~~~~~~~~~^^
[rank1]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/trainer.py", line 264, in train
[rank1]:     self._do_train()
[rank1]:     ~~~~~~~~~~~~~~^^
[rank1]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/trainer.py", line 522, in _do_train
[rank1]:     for i, batch in pbar:
[rank1]:                     ^^^^
[rank1]:   File "

KeyboardInterrupt: 

In [10]:
from pathlib import Path
import pandas as pd

# Actual training output directory
RUN_DIR = Path(
    "/kaggle/working/YOLO_Experiments/baseline_yolov8n_sgd-2"
)

BEST_PATH = RUN_DIR / "weights" / "best.pt"
LAST_PATH = RUN_DIR / "weights" / "last.pt"
RESULTS_PATH = RUN_DIR / "results.csv"

print("========== CHECKPOINT VERIFICATION ==========")

print("\nRun directory:", RUN_DIR)
print("Directory exists:", RUN_DIR.exists())

# 1. Verify important files
print("\n--- SAVED FILES ---")

for file in [BEST_PATH, LAST_PATH, RESULTS_PATH]:
    if file.exists():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"✓ {file.name}: {size_mb:.2f} MB")
    else:
        print(f"✗ {file.name}: NOT FOUND")

# 2. Check all model checkpoints
print("\n--- ALL CHECKPOINTS ---")

weights_dir = RUN_DIR / "weights"

if weights_dir.exists():
    for file in sorted(weights_dir.glob("*.pt")):
        print(file.name)

# 3. Verify completed epochs and metrics
if RESULTS_PATH.exists():
    df = pd.read_csv(RESULTS_PATH)
    df.columns = df.columns.str.strip()

    print("\nCompleted epochs:", len(df))

    print("\n--- LAST 5 EPOCHS ---")
    display(df.tail())

    best_row = df.loc[
        df["metrics/mAP50-95(B)"].idxmax()
    ]

    print("\n--- BEST VALIDATION RESULT ---")
    print("Epoch:", int(best_row["epoch"]))
    print("Precision:", best_row["metrics/precision(B)"])
    print("Recall:", best_row["metrics/recall(B)"])
    print("mAP50:", best_row["metrics/mAP50(B)"])
    print("mAP50-95:", best_row["metrics/mAP50-95(B)"])

else:
    print("\nresults.csv NOT FOUND!")

========== CHECKPOINT VERIFICATION ==========

Run directory: /kaggle/working/YOLO_Experiments/baseline_yolov8n_sgd-2
Directory exists: True

--- SAVED FILES ---
✓ best.pt: 12.26 MB
✓ last.pt: 12.26 MB
✓ results.csv: 0.00 MB

--- ALL CHECKPOINTS ---
best.pt
epoch0.pt
epoch10.pt
epoch15.pt
epoch5.pt
last.pt

Completed epochs: 16

--- LAST 5 EPOCHS ---


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
11,12,3477.93,1.89777,2.09203,1.72731,0.47247,0.38871,0.37527,0.16669,1.87950,1.99778,1.71365,0.008864,0.008864,0.008864
12,13,3741.82,1.89178,2.06093,1.70834,0.48381,0.38851,0.38370,0.17488,1.85471,1.95515,1.67882,0.008658,0.008658,0.008658
13,14,4002.17,1.86215,2.03669,1.69738,0.49163,0.39037,0.38923,0.17815,1.85065,1.94169,1.68490,0.008439,0.008439,0.008439
14,15,4262.68,1.85628,1.99618,1.68621,0.48796,0.39850,0.39600,0.18152,1.82640,1.91843,1.66759,0.008205,0.008205,0.008205
15,16,4522.93,1.85228,1.97263,1.67894,0.51169,0.41295,0.41924,0.19242,1.81063,1.86243,1.63545,0.007960,0.007960,0.007960



--- BEST VALIDATION RESULT ---
Epoch: 16
Precision: 0.51169
Recall: 0.41295
mAP50: 0.41924
mAP50-95: 0.19242


In [11]:
from pathlib import Path
import shutil

RUN_DIR = Path(
    "/kaggle/working/YOLO_Experiments/baseline_yolov8n_sgd-2"
)

BACKUP_PATH = "/kaggle/working/SGD_Baseline_16_Epochs"

assert (RUN_DIR / "weights/best.pt").exists()
assert (RUN_DIR / "weights/last.pt").exists()
assert (RUN_DIR / "results.csv").exists()

# Create ZIP backup
zip_path = shutil.make_archive(
    BACKUP_PATH,
    "zip",
    root_dir=RUN_DIR
)

size_mb = Path(zip_path).stat().st_size / (1024 ** 2)

print("========== BASELINE BACKUP ==========")
print("Backup created:", zip_path)
print("Backup size:", round(size_mb, 2), "MB")
print("Backup exists:", Path(zip_path).exists())
print("\nBaseline training outputs archived successfully!")

========== BASELINE BACKUP ==========
Backup created: /kaggle/working/SGD_Baseline_16_Epochs.zip
Backup size: 68.92 MB
Backup exists: True

Baseline training outputs archived successfully!


In [13]:
from IPython.display import FileLink, display
import os

os.chdir("/kaggle/working")

display(FileLink("SGD_Baseline_16_Epochs.zip"))

/kaggle/working/SGD_Baseline_16_Epochs.zip